# Qwen Base Model: MLP Tensor Disentanglement and MMLU

Loads `Qwen/Qwen3.5-0.8B-Base`, prints compact model structure summaries, runs a baseline MMLU evaluation, applies tensor disentanglement to selected MLP linear layers, then reruns MMLU and compares the scores.

In [ ]:
import contextlib
import gc
import io
import logging
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"

# Override this if needed: "cuda", "mps", or "cpu".
REQUESTED_DEVICE = None

if REQUESTED_DEVICE is not None:
    DEVICE = REQUESTED_DEVICE
elif torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

if DEVICE == "cuda":
    TORCH_DTYPE = torch.bfloat16
elif DEVICE == "mps":
    # fp32 + lm-eval auto batching can exceed unified memory on Apple Silicon.
    TORCH_DTYPE = torch.float16
else:
    TORCH_DTYPE = torch.float32

print(f"Loading {MODEL_NAME} on {DEVICE} with dtype={TORCH_DTYPE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs = {
    "dtype": TORCH_DTYPE,
    "trust_remote_code": True,
}
if DEVICE == "cuda":
    load_kwargs["device_map"] = "cuda"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
if DEVICE != "cuda":
    model = model.to(DEVICE)

model.eval()
model.config.use_cache = False
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()

print(f"Loaded model class: {model.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from collections import Counter

import torch.nn as nn
from IPython.display import Markdown, display


def count_params(module, *, recurse=True):
    return sum(param.numel() for param in module.parameters(recurse=recurse))


def fmt_int(value):
    return f"{value:,}" if isinstance(value, int) else value


def md_table(headers, rows):
    rows = [[str(cell) for cell in row] for row in rows]
    header = "| " + " | ".join(headers) + " |"
    sep = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, sep, *body])


layers = list(model.model.layers)
layer_types = getattr(model.config, "layer_types", [getattr(layer, "layer_type", "unknown") for layer in layers])
layer_type_counts = Counter(layer_types)

summary_rows = [
    ["model", MODEL_NAME],
    ["class", model.__class__.__name__],
    ["device", DEVICE],
    ["dtype", str(TORCH_DTYPE).replace("torch.", "")],
    ["parameters", fmt_int(count_params(model))],
    ["hidden size", getattr(model.config, "hidden_size", None)],
    ["intermediate size", getattr(model.config, "intermediate_size", None)],
    ["decoder layers", len(layers)],
    ["attention heads", getattr(model.config, "num_attention_heads", None)],
    ["vocab size", fmt_int(getattr(model.config, "vocab_size", None))],
    ["layer types", ", ".join(f"{name}: {count}" for name, count in layer_type_counts.items())],
]
display(Markdown("## Model summary\n" + md_table(["field", "value"], summary_rows)))

top_modules = [
    ["model.embed_tokens", model.model.embed_tokens.__class__.__name__, fmt_int(count_params(model.model.embed_tokens))],
    ["model.layers", model.model.layers.__class__.__name__, fmt_int(count_params(model.model.layers))],
    ["model.norm", model.model.norm.__class__.__name__, fmt_int(count_params(model.model.norm))],
    ["lm_head", model.lm_head.__class__.__name__, fmt_int(count_params(model.lm_head))],
]
display(Markdown("## Top-level modules\n" + md_table(["module", "type", "params"], top_modules)))

layer_rows = []
for idx, layer in enumerate(layers):
    if hasattr(layer, "linear_attn"):
        mixer_name = "linear_attn"
        mixer = layer.linear_attn
    elif hasattr(layer, "self_attn"):
        mixer_name = "self_attn"
        mixer = layer.self_attn
    else:
        mixer_name = "unknown"
        mixer = layer
    layer_rows.append([
        idx,
        layer_types[idx] if idx < len(layer_types) else getattr(layer, "layer_type", "unknown"),
        mixer_name,
        mixer.__class__.__name__,
        fmt_int(count_params(mixer)),
        fmt_int(count_params(layer.mlp)),
    ])
display(Markdown("## Decoder layers\n" + md_table(["idx", "layer type", "mixer", "mixer class", "mixer params", "mlp params"], layer_rows)))

mlp_shape_counts = Counter()
for layer in layers:
    for name, submodule in layer.mlp.named_modules():
        if isinstance(submodule, nn.Linear):
            mlp_shape_counts[(name, tuple(submodule.weight.shape), submodule.bias is not None)] += 1

mlp_rows = [
    [name, f"{shape[0]} x {shape[1]}", has_bias, count]
    for (name, shape, has_bias), count in sorted(mlp_shape_counts.items())
]
display(Markdown("## MLP linear shapes\n" + md_table(["MLP submodule", "weight shape", "bias", "count"], mlp_rows)))

In [ ]:
import gc

from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

NUM_FEWSHOT = 5
EVAL_LIMIT = 5  # Use None for the full MMLU run once the smoke test works.
TASKS = ["mmlu"]
EVAL_BATCH_SIZE = 1
MAX_LENGTH = 2048

EVAL_RUNS = {}


def clear_device_cache():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def first_metric(metrics, names):
    for name in names:
        if name in metrics and isinstance(metrics[name], (int, float)):
            return float(metrics[name])
    return None


def extract_mmlu_scores(results):
    result_dict = results["results"]
    if "mmlu" in result_dict:
        metrics = result_dict["mmlu"]
        return {
            "acc": first_metric(metrics, ["acc,none", "acc"]),
            "acc_stderr": first_metric(metrics, ["acc_stderr,none", "acc_stderr"]),
            "acc_norm": first_metric(metrics, ["acc_norm,none", "acc_norm"]),
            "acc_norm_stderr": first_metric(metrics, ["acc_norm_stderr,none", "acc_norm_stderr"]),
            "num_tasks": 1,
        }

    acc_values = []
    acc_norm_values = []
    for metrics in result_dict.values():
        if not isinstance(metrics, dict):
            continue
        acc = first_metric(metrics, ["acc,none", "acc"])
        acc_norm = first_metric(metrics, ["acc_norm,none", "acc_norm"])
        if acc is not None:
            acc_values.append(acc)
        if acc_norm is not None:
            acc_norm_values.append(acc_norm)
    return {
        "acc": sum(acc_values) / len(acc_values) if acc_values else None,
        "acc_stderr": None,
        "acc_norm": sum(acc_norm_values) / len(acc_norm_values) if acc_norm_values else None,
        "acc_norm_stderr": None,
        "num_tasks": len(result_dict),
    }


def fmt_float(value):
    return "n/a" if value is None else f"{value:.4f}"


def display_eval_scores(label, scores):
    rows = [
        ["acc", fmt_float(scores["acc"])],
        ["acc stderr", fmt_float(scores["acc_stderr"])],
        ["acc norm", fmt_float(scores["acc_norm"])],
        ["acc norm stderr", fmt_float(scores["acc_norm_stderr"])],
        ["reported tasks", scores["num_tasks"]],
    ]
    display(Markdown(f"## {label} MMLU summary\n" + md_table(["metric", "value"], rows)))


def run_mmlu_eval(label):
    clear_device_cache()
    lm = HFLM(
        pretrained=model,
        tokenizer=tokenizer,
        batch_size=EVAL_BATCH_SIZE,
        max_batch_size=EVAL_BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
        dtype=TORCH_DTYPE,
        trust_remote_code=True,
    )
    print(f"Running {label} MMLU: tasks={TASKS}, fewshot={NUM_FEWSHOT}, limit={EVAL_LIMIT}, batch={EVAL_BATCH_SIZE}")
    captured_stdout = io.StringIO()
    captured_stderr = io.StringIO()
    previous_logging_disable_level = logging.root.manager.disable
    try:
        logging.disable(logging.WARNING)
        with contextlib.redirect_stdout(captured_stdout), contextlib.redirect_stderr(captured_stderr):
            results = evaluator.simple_evaluate(
                model=lm,
                tasks=TASKS,
                num_fewshot=NUM_FEWSHOT,
                limit=EVAL_LIMIT,
                log_samples=False,
                bootstrap_iters=1000,
                verbosity="ERROR",
            )
    finally:
        logging.disable(previous_logging_disable_level)
    del lm
    clear_device_cache()
    scores = extract_mmlu_scores(results)
    EVAL_RUNS[label] = {"results": results, "scores": scores}
    display_eval_scores(label, scores)
    return results


baseline_results = run_mmlu_eval("baseline")

In [ ]:
import contextlib
import io
import time

import tensor_disentanglement_code.disentanglement_base as DB


# Full MLP pass. Set to a smaller list such as [0, 1, 2, 3] for a quicker trial.
DISENTANGLE_LAYER_INDICES = None
DISENTANGLE_MLP_PROJECTIONS = ("gate_proj", "up_proj", "down_proj")
DISENTANGLE_RANK_FRACTION = 0.25
DISENTANGLE_STEPS = 3
DISENTANGLE_LEARNING_RATE = 0.05
DISENTANGLE_OPTIMIZER = "alt"
DISENTANGLE_SEED = 0


def shape_tuple_from_result(result):
    shape = result.tensor_shape_4d
    return (shape.left_outer, shape.left_inner, shape.right_outer, shape.right_inner)


def shape_tuple_from_weight(weight):
    shape = DB.choose_tensor_shape(weight)
    return (shape.left_outer, shape.left_inner, shape.right_outer, shape.right_inner)


def direct_truncated_svd_weight(weight, keep_rank):
    u, s, vh = torch.linalg.svd(weight, full_matrices=False)
    truncated = (u[:, :keep_rank] * s[:keep_rank]) @ vh[:keep_rank]
    rel_error = torch.linalg.norm(truncated - weight) / torch.linalg.norm(weight)
    return truncated, float(rel_error.item())


def disentangled_dense_weight(linear, *, rank_fraction, steps, learning_rate, optimizer_type, seed):
    original_weight = linear.weight.detach().cpu().to(dtype=torch.float64)
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stderr(stderr_buffer):
        result = DB.disentangle_matrix_via_reshape(
            original_weight,
            steps=steps,
            learning_rate=learning_rate,
            seed=seed,
            device="cpu",
            optimizer_type=optimizer_type,
            rank_selection="fixed_heuristic",
            rank_fraction=rank_fraction,
            log_every=0,
        )
    keep_rank = result.inner.keep_rank
    shape_tuple = shape_tuple_from_result(result)
    q = result.inner.best_q.to(dtype=torch.float64)
    permuted = DB._apply_A(q @ original_weight, shape_tuple)
    u, s, vh = torch.linalg.svd(permuted, full_matrices=False)
    truncated = (u[:, :keep_rank] * s[:keep_rank]) @ vh[:keep_rank]
    reconstructed = q.T @ DB._apply_A_inv(truncated, shape_tuple)
    rel_error = torch.linalg.norm(reconstructed - original_weight) / torch.linalg.norm(original_weight)
    return reconstructed, result, float(rel_error.item()), "tensor_disentanglement"


def selected_mlp_targets(module):
    all_layers = list(module.model.layers)
    layer_indices = DISENTANGLE_LAYER_INDICES
    if layer_indices is None:
        layer_indices = list(range(len(all_layers)))

    targets = []
    for layer_idx in layer_indices:
        mlp = all_layers[layer_idx].mlp
        for projection_name in DISENTANGLE_MLP_PROJECTIONS:
            projection = getattr(mlp, projection_name, None)
            if projection is not None:
                targets.append((layer_idx, projection_name, projection))
    return targets


def apply_mlp_tensor_disentanglement(module):
    targets = selected_mlp_targets(module)
    rows = []
    start = time.time()
    module.eval()
    clear_device_cache()
    print(
        f"Disentangling {len(targets)} MLP linear layers "
        f"with rank_fraction={DISENTANGLE_RANK_FRACTION}, steps={DISENTANGLE_STEPS}, "
        f"optimizer={DISENTANGLE_OPTIMIZER}."
    )

    with torch.inference_mode():
        for target_idx, (layer_idx, projection_name, projection) in enumerate(targets):
            target_name = f"model.layers.{layer_idx}.mlp.{projection_name}"
            print(f"[{target_idx + 1:02d}/{len(targets):02d}] {target_name}")
            target_start = time.time()
            try:
                new_weight, result, rel_error, method = disentangled_dense_weight(
                    projection,
                    rank_fraction=DISENTANGLE_RANK_FRACTION,
                    steps=DISENTANGLE_STEPS,
                    learning_rate=DISENTANGLE_LEARNING_RATE,
                    optimizer_type=DISENTANGLE_OPTIMIZER,
                    seed=DISENTANGLE_SEED + target_idx,
                )
                keep_rank = result.inner.keep_rank
                initial_effective_rank = result.inner.initial_stats.effective_rank
                best_effective_rank = result.inner.best_stats.effective_rank
                error = ""
            except torch.linalg.LinAlgError as exc:
                original_weight = projection.weight.detach().cpu().to(dtype=torch.float64)
                shape_tuple = shape_tuple_from_weight(original_weight)
                keep_rank = DB._default_keep_rank_from_shape(shape_tuple, DISENTANGLE_RANK_FRACTION)
                new_weight, rel_error = direct_truncated_svd_weight(original_weight, keep_rank)
                initial_effective_rank = None
                best_effective_rank = None
                method = "direct_svd_fallback"
                error = f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
            copied_weight = new_weight.to(dtype=projection.weight.dtype).to(device=projection.weight.device)
            projection.weight.copy_(copied_weight)
            rows.append({
                "target": target_name,
                "layer": layer_idx,
                "projection": projection_name,
                "shape": tuple(projection.weight.shape),
                "keep_rank": keep_rank,
                "initial_effective_rank": initial_effective_rank,
                "best_effective_rank": best_effective_rank,
                "relative_error": rel_error,
                "method": method,
                "error": error,
                "seconds": time.time() - target_start,
            })
            clear_device_cache()

    projection_rows = []
    for projection_name in DISENTANGLE_MLP_PROJECTIONS:
        projection_items = [row for row in rows if row["projection"] == projection_name]
        if not projection_items:
            continue
        projection_rows.append([
            projection_name,
            len(projection_items),
            f"{sum(row['relative_error'] for row in projection_items) / len(projection_items):.4f}",
            f"{sum(row['best_effective_rank'] for row in projection_items if row['best_effective_rank'] is not None) / max(1, sum(row['best_effective_rank'] is not None for row in projection_items)):.2f}",
            ", ".join(f"{method}: {sum(row['method'] == method for row in projection_items)}" for method in sorted({row['method'] for row in projection_items})),
            f"{sum(row['seconds'] for row in projection_items):.1f}s",
        ])

    detail_rows = [
        [
            row["target"],
            f"{row['shape']}",
            row["keep_rank"],
            "n/a" if row['initial_effective_rank'] is None else f"{row['initial_effective_rank']:.2f}",
            "n/a" if row['best_effective_rank'] is None else f"{row['best_effective_rank']:.2f}",
            f"{row['relative_error']:.4f}",
            row["method"],
            row["error"],
            f"{row['seconds']:.1f}s",
        ]
        for row in rows
    ]

    display(Markdown(
        "## MLP tensor disentanglement summary\n"
        + f"Processed {len(rows)} MLP linear layers in {time.time() - start:.1f}s. "
        + f"Rank fraction: `{DISENTANGLE_RANK_FRACTION}`, steps: `{DISENTANGLE_STEPS}`, optimizer: `{DISENTANGLE_OPTIMIZER}`.\n\n"
        + md_table(
            ["projection", "count", "mean rel error", "mean best eff rank", "methods", "total time"],
            projection_rows,
        )
        + "\n\n<details><summary>Per-layer details</summary>\n\n"
        + md_table(
            ["target", "shape", "keep rank", "initial eff rank", "best eff rank", "rel error", "method", "error", "time"],
            detail_rows,
        )
        + "\n\n</details>"
    ))
    return rows


disentanglement_report = apply_mlp_tensor_disentanglement(model)

In [ ]:
mlp_disentangled_results = run_mmlu_eval("mlp_disentangled")


def compare_eval_runs(left_label="baseline", right_label="mlp_disentangled"):
    left = EVAL_RUNS[left_label]["scores"]
    right = EVAL_RUNS[right_label]["scores"]
    rows = []
    for metric in ["acc", "acc_norm"]:
        left_value = left.get(metric)
        right_value = right.get(metric)
        if left_value is None or right_value is None:
            delta = None
        else:
            delta = right_value - left_value
        rows.append([metric, fmt_float(left_value), fmt_float(right_value), fmt_float(delta)])
    display(Markdown(
        "## Baseline vs MLP-disentangled MMLU\n"
        + md_table(["metric", left_label, right_label, "delta"], rows)
    ))


compare_eval_runs()